## Data Loader

In [23]:
import tiktoken

In [24]:
# Create ticktoken BPE tokenizer
tokenizer = tiktoken.get_encoding("gpt2")

In [25]:
# Read text file
SENTENCES_FILE_PATH = "../data/the-verdict.txt"

with open(SENTENCES_FILE_PATH, "r", encoding="utf-8") as f:
    raw_text = f.read()

In [26]:
# Tokenize text
txt = tokenizer.encode(raw_text)
print("Number of tokens: ", len(txt))

Number of tokens:  5145


In [27]:
# Take a sample
sample = txt[1010:1055]
print(f"Token IDs: ", sample)
print(f"Tokens: ")

for token_id in sample:
    print(tokenizer.decode([token_id]), end="|")

Token IDs:  [35924, 262, 12306, 395, 4133, 13, 198, 198, 1, 26788, 338, 691, 12226, 318, 284, 1234, 8737, 656, 19133, 553, 373, 530, 286, 262, 7877, 72, 3150, 339, 8104, 866, 1973, 262, 37918, 411, 290, 8465, 286, 281, 33954, 271, 3973, 9899, 14678, 40556, 12]
Tokens: 
poke| the| ampl|est| resources|.|
|
|"|Money|'s| only| excuse| is| to| put| beauty| into| circulation|,"| was| one| of| the| ax|i|oms| he| laid| down| across| the| Sev|res| and| silver| of| an| exqu|is|itely| appointed| lun|cheon|-|

In [28]:
context_size = 4 # How many tokens do we look at when predicting the next token?
x = txt[:context_size] # Input tokens
y = txt[1:context_size+1] # Target token (the one we want to predict)
print("x: ", x)
print("y:     ", y)

x:  [40, 367, 2885, 1464]
y:      [367, 2885, 1464, 1807]


In [29]:
print("[input token IDs] -> target token ID")
print("-------------------------------")

for i in range(1, context_size+1):
    context = sample[:i]
    desired = sample[i]
    print(f"{context} -> {desired}")

[input token IDs] -> target token ID
-------------------------------
[35924] -> 262
[35924, 262] -> 12306
[35924, 262, 12306] -> 395
[35924, 262, 12306, 395] -> 4133


In [30]:
print("[input tokens] -> target token")
print("-------------------------------")

for i in range(1, context_size+1):
    context = sample[:i]
    desired = sample[i]
    print(f"{tokenizer.decode(context)} -> {tokenizer.decode([desired])}")

[input tokens] -> target token
-------------------------------
poke ->  the
poke the ->  ampl
poke the ampl -> est
poke the amplest ->  resources


In [31]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDataset1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt)
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i+max_length]
            target_chunk = token_ids[i+1:i+max_length+1]

            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        """Returns the number of samples in the dataset."""
        return len(self.input_ids)

    def __getitem__(self, idx):
        """Returns a single sample from the dataset."""
        return self.input_ids[idx], self.target_ids[idx]

In [32]:
# Demonstration of GPTDataset1 with dummy data.
example_token_ids = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]

max_length = 4
stride = 1

print("[input token IDs chunk] -> [target token IDs chunk]")
print("---------------------------------------")

for i in range(0, len(example_token_ids) - max_length, stride):
    input_chunk = example_token_ids[i:i+max_length]
    target_chunk = example_token_ids[i+1:i+max_length+1]
    print(f"{input_chunk} -> {target_chunk}")

[input token IDs chunk] -> [target token IDs chunk]
---------------------------------------
[0, 1, 2, 3] -> [1, 2, 3, 4]
[1, 2, 3, 4] -> [2, 3, 4, 5]
[2, 3, 4, 5] -> [3, 4, 5, 6]
[3, 4, 5, 6] -> [4, 5, 6, 7]
[4, 5, 6, 7] -> [5, 6, 7, 8]
[5, 6, 7, 8] -> [6, 7, 8, 9]
[6, 7, 8, 9] -> [7, 8, 9, 10]
[7, 8, 9, 10] -> [8, 9, 10, 11]
[8, 9, 10, 11] -> [9, 10, 11, 12]
[9, 10, 11, 12] -> [10, 11, 12, 13]
[10, 11, 12, 13] -> [11, 12, 13, 14]
[11, 12, 13, 14] -> [12, 13, 14, 15]
[12, 13, 14, 15] -> [13, 14, 15, 16]
[13, 14, 15, 16] -> [14, 15, 16, 17]
[14, 15, 16, 17] -> [15, 16, 17, 18]
[15, 16, 17, 18] -> [16, 17, 18, 19]


In [33]:
def create_dataloader_v1(txt: str,
                         batch_size: int = 4,
                         max_length: int = 256,
                         stride: int = 128,
                         shuffle: bool = True,
                         drop_last: bool = True,
                         num_workers: int = 0):
    """Creates a torch.utilsdata.DataLoader
    
    Args:
        txt: The input text to be tokenized and used for training.
        batch_size: The number of samples per batch.
        max_length: The maximum length of input sequences.
        stride: The step size for creating overlapping sequences.
        shuffle: Whether to shuffle the dataset.
        drop_last : Whether to drop the last batch if it is shorter than batch_size. Prevents loss spikes during training.
        num_workers: The number of subprocesses to use for preprocessing.

        Returns:    A DataLoader that yields batches of (input_ids, target_ids) tuples.
    """

    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDataset1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last =drop_last,
        num_workers=num_workers
    )

    return dataloader

In [38]:
with open(SENTENCES_FILE_PATH, "r", encoding="utf-8") as f:
    raw_text = f.read()

    dataloader = create_dataloader_v1(txt=raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
    data_iter = iter(dataloader)
    first_batch = next(data_iter)

In [39]:
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [40]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [42]:
dataloader2 = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

In [43]:
data_iter2 = iter(dataloader2)
inputs, targets = next(data_iter2)
print("Inputs:\n", inputs)
print("Targets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])
